# 077: KmerSeek hp_pbotc_1st_ed k=26 — QFO 2020 partial analysis

Exploratory analysis of the in-progress `hp-pbotc-1st-ed` k=26 all-vs-all run on QfO 2020 (78 proteomes, 3,003 pairs).

Files come from:
`nextflow-runs/qfo/2020-pbotc-k26/results/search_results/`

Each CSV is unfiltered (all pairs with ≥1 shared k-mer). Files can be tens of millions of rows,
so this notebook aggregates per-file rather than loading everything at once.

In [1]:
import glob
import os
import re

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import polars as pl
from tqdm.notebook import tqdm

RESULTS_DIR = os.path.expanduser(
    '~/code/2024-kmerseek-analysis/nextflow-runs/qfo/2020-pbotc-k26/results/search_results'
)
TOTAL_PAIRS = 3003  # 78 * 77 / 2

PVAL_THRESHOLDS = [0.05, 1e-5, 1e-10, 1e-20]

# taxid → common name for labelling
TAXID_LABELS = {
    '9606':   'Human',
    '10090':  'Mouse',
    '7955':   'Zebrafish',
    '9031':   'Chicken',
    '7227':   'Fly',
    '6239':   'Worm',
    '559292': 'Yeast',
    '3702':   'Arabidopsis',
    '83333':  'E.coli',
    '7719':   'Ciona',
}

In [ ]:
csvs = sorted(glob.glob(os.path.join(RESULTS_DIR, '*.csv.gz')))
print(f'Files found: {len(csvs)} / {TOTAL_PAIRS} pairs ({100*len(csvs)/TOTAL_PAIRS:.1f}% done)')

records = []
for f in tqdm(csvs, desc='Scanning files', unit='pair'):
    size = os.path.getsize(f)
    base = os.path.basename(f).replace('.k26.csv.gz', '')
    q_prot, t_prot = base.split('_vs_')
    q_tax = q_prot.rsplit('_', 1)[1]
    t_tax = t_prot.rsplit('_', 1)[1]

    rec = dict(query_proteome=q_prot, target_proteome=t_prot,
               query_taxid=q_tax, target_taxid=t_tax,
               file_bytes=size)

    if size < 50:
        for t in PVAL_THRESHOLDS:
            rec[f'hits_p{t:.0e}'] = 0
        rec.update(total_rows=0, n_queries=0, n_targets=0,
                   bonferroni_threshold=np.nan, hits_bonferroni=0)
        records.append(rec)
        continue

    try:
        # Load 3 columns once — avoids scanning the file twice (was the bottleneck)
        df = pl.scan_csv(f).select(['query_name', 'target_name', 'poisson_pvalue']).collect()
        pvals = df['poisson_pvalue']
        n_queries = df['query_name'].n_unique()
        n_targets = df['target_name'].n_unique()
        total_rows = len(df)
        n_tests = n_queries * n_targets
        bf_thresh = 0.05 / n_tests if n_tests > 0 else float('nan')

        rec.update({
            'total_rows':          total_rows,
            'n_queries':           n_queries,
            'n_targets':           n_targets,
            'bonferroni_threshold': bf_thresh,
            **{f'hits_p{t:.0e}': int((pvals < t).sum()) for t in PVAL_THRESHOLDS},
            'hits_bonferroni': int((pvals < bf_thresh).sum()) if not np.isnan(bf_thresh) else 0,
        })

    except Exception as e:
        tqdm.write(f'  ERROR {base}: {e}')
        rec.update(total_rows=-1, n_queries=-1, n_targets=-1,
                   bonferroni_threshold=np.nan, hits_bonferroni=-1)
        for t in PVAL_THRESHOLDS:
            rec[f'hits_p{t:.0e}'] = -1

    records.append(rec)

stats = pd.DataFrame(records)
total_rows = stats['total_rows'].sum()
print(f'\nTotal raw hits across all loaded pairs: {total_rows:,}')
for t in PVAL_THRESHOLDS:
    col = f'hits_p{t:.0e}'
    n = stats[col].sum()
    print(f'  p < {t:.0e} (raw):        {n:,}  ({100*n/total_rows:.2f}%)')
n_bf = stats['hits_bonferroni'].sum()
print(f'  Bonferroni p<0.05:       {n_bf:,}  ({100*n_bf/total_rows:.2f}%)')
stats.head(3)


Files found: 1380 / 3003 pairs (46.0% done)


Scanning files:   0%|          | 0/1380 [00:00<?, ?pair/s]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# File sizes
ax = axes[0]
nonempty = stats[stats['total_rows'] > 0]
ax.hist(nonempty['file_bytes'] / 1e6, bins=40, color='steelblue', edgecolor='white')
ax.set_xlabel('Compressed file size (MB)')
ax.set_ylabel('Pairs')
ax.set_title(f'Search result file sizes\n({len(nonempty)} pairs, hp_pbotc_1st_ed k=26)')

# Raw rows per pair
ax = axes[1]
ax.hist(np.log10(nonempty['total_rows'].clip(lower=1)), bins=40, color='darkorange', edgecolor='white')
ax.set_xlabel('log₁₀(raw hits per pair)')
ax.set_ylabel('Pairs')
ax.set_title('Raw hit counts per species pair')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'10^{x:.0f}'))

plt.tight_layout()
plt.show()

In [ ]:
nonempty = stats[stats['total_rows'] > 0]

plot_cols = [(f'hits_p{t:.0e}', f'p < {t:.0e} (raw)') for t in PVAL_THRESHOLDS]
plot_cols.append(('hits_bonferroni', 'Bonferroni\np < 0.05'))

fig, axes = plt.subplots(1, len(plot_cols), figsize=(18, 4), sharey=False)

colors = ['#4878CF', '#6ACC65', '#D65F5F', '#B47CC7', 'mediumpurple']
for ax, (col, title), color in zip(axes, plot_cols, colors):
    vals = nonempty[col]
    nonzero = vals[vals > 0]
    if len(nonzero) == 0:
        ax.text(0.5, 0.5, 'no hits', ha='center', va='center', transform=ax.transAxes)
    else:
        ax.hist(np.log10(nonzero.clip(lower=1)), bins=30, color=color, edgecolor='white')
        ax.set_title(f'{title}\nn={nonzero.sum():,} total\nmedian {nonzero.median():.0f}/pair')
    ax.set_xlabel('log₁₀(hits per pair)')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'$10^{{{x:.0f}}}$'))
    ax.set_ylabel('Pairs')

plt.suptitle('Hits per pair — hp_pbotc_1st_ed k=26', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
def label(taxid):
    return TAXID_LABELS.get(taxid, taxid)

top = (
    stats[stats['hits_bonferroni'] > 0]
    .assign(pair=lambda d: d['query_taxid'].map(label) + ' vs ' + d['target_taxid'].map(label))
    .nlargest(20, 'hits_bonferroni')[['pair', 'total_rows', 'hits_bonferroni', 'bonferroni_threshold']]
    .reset_index(drop=True)
)
top.columns = ['Pair', 'Raw hits', 'Bonferroni hits', 'BF threshold']
top['% significant'] = (100 * top['Bonferroni hits'] / top['Raw hits']).round(2)
top['BF threshold'] = top['BF threshold'].map('{:.2e}'.format)
print('Top 20 pairs by Bonferroni-corrected hits (p < 0.05 / n_tests):')
top

In [ ]:
# Detailed look at one pair: pick smallest file with most hits at p < 1e-5
col = 'hits_p1e-05'
candidate = (
    stats[stats[col] > 0]
    .nsmallest(10, 'file_bytes')
    .nlargest(1, col)
    .iloc[0]
)
sample_file = os.path.join(
    RESULTS_DIR,
    f"{candidate['query_proteome']}_vs_{candidate['target_proteome']}.k26.csv.gz"
)
print(f'Sampling: {os.path.basename(sample_file)}')
print(f'  {label(candidate["query_taxid"])} vs {label(candidate["target_taxid"])}')
print(f'  File size: {candidate["file_bytes"]/1e6:.0f} MB, total rows: {candidate["total_rows"]:,}')

# Only load significant rows
sample = (
    pl.scan_csv(sample_file)
    .select(['query_name', 'target_name', 'jaccard', 'max_containment',
             'poisson_pvalue', 'n_intersecting_hashes'])
    .filter(pl.col('poisson_pvalue') < 0.05)
    .collect()
)
print(f'  Rows loaded (p<0.05): {len(sample):,}')
print(f'  Rows at p<1e-5: {(sample["poisson_pvalue"] < 1e-5).sum():,}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

pair_label = f"{label(candidate['query_taxid'])} vs {label(candidate['target_taxid'])}"

ax = axes[0]
pvals = sample['poisson_pvalue'].to_numpy()
ax.hist(np.log10(pvals.clip(min=1e-300)), bins=60, color='steelblue', edgecolor='white')
ax.axvline(np.log10(1e-5), color='red', linestyle='--', label='p=1e-5')
ax.set_xlabel('log₁₀(poisson p-value)')
ax.set_ylabel('Protein pairs')
ax.set_title(f'P-value distribution\n{pair_label}')
ax.legend()

ax = axes[1]
jac = sample['jaccard'].to_numpy()
ax.hist(np.log10(jac.clip(min=1e-10)), bins=60, color='darkorange', edgecolor='white')
ax.set_xlabel('log₁₀(Jaccard)')
ax.set_ylabel('Protein pairs')
ax.set_title(f'Jaccard distribution\n{pair_label}')

ax = axes[2]
sig = sample.filter(pl.col('poisson_pvalue') < 1e-5)
ax.hist(sig['jaccard'].to_numpy(), bins=60, color='mediumpurple', edgecolor='white')
ax.set_xlabel('Jaccard')
ax.set_ylabel('Protein pairs')
ax.set_title(f'Jaccard (p<1e-5 hits only)\nn={len(sig):,}')

plt.suptitle(f'hp_pbotc_1st_ed k=26', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
done = len(stats)
remaining = TOTAL_PAIRS - done
print(f'Pipeline progress: {done}/{TOTAL_PAIRS} pairs ({100*done/TOTAL_PAIRS:.1f}%)')
print(f'Remaining: {remaining} pairs')
print()
print('Re-run this notebook after the pipeline completes for full analysis.')